# TUM-VIE New Models Ablation

Ablation study for newly added fusion models on the same split.

Models compared:
- IMUConditionedVisualSNN
- VisualIMURecurrentFusionBlock
- KalmanStyleSpikingFusionSurrogate

In [ ]:
from pathlib import Path

import haiku as hk
import jax
import jax.numpy as jnp
import numpy as np
import optax
from sklearn.model_selection import train_test_split
from tonic import transforms
from tonic.datasets import TUMVIE

import spyx.models as fm

SEED = 11
RECORDING = "mocap-6dof"
SPATIAL_FACTOR = 0.2
N_TIME_BINS = 20
N_SAMPLES = 150
BATCH_SIZE = 8
EPOCHS = 3

data_root = Path("../data").resolve()
sensor = TUMVIE.sensor_size
sensor_small = (int(sensor[0] * SPATIAL_FACTOR), int(sensor[1] * SPATIAL_FACTOR), sensor[2])

tfm = transforms.Compose([
    transforms.Downsample(spatial_factor=SPATIAL_FACTOR),
    transforms.ToFrame(sensor_size=sensor_small, n_time_bins=N_TIME_BINS),
])

ds = TUMVIE(save_to=str(data_root), recording=RECORDING, transform=tfm)
print("dataset size =", len(ds))

In [ ]:
def build_arrays(limit=N_SAMPLES):
    xs, us, ys = [], [], []
    n = min(limit, len(ds))
    for i in range(n):
        d, t = ds[i]
        left = np.asarray(d["events_left"], dtype=np.float32)
        right = np.asarray(d["events_right"], dtype=np.float32)
        x = np.concatenate([left, right], axis=1)

        imu = np.asarray(d["imu"], dtype=np.float32)
        imu = imu.mean(axis=0) if imu.ndim == 2 and imu.shape[0] > 0 else np.zeros((6,), dtype=np.float32)

        mocap = np.asarray(t["mocap"], dtype=np.float32)
        mocap = mocap[-1] if mocap.ndim == 2 else mocap
        y = mocap[:3]

        xs.append(x)
        us.append(imu)
        ys.append(y)

    return np.stack(xs, axis=0), np.stack(us, axis=0), np.stack(ys, axis=0)


X, U, Y = build_arrays()
idx = np.arange(X.shape[0])
tr, te = train_test_split(idx, test_size=0.2, random_state=SEED, shuffle=True)
tr, va = train_test_split(tr, test_size=0.2, random_state=SEED + 1, shuffle=True)

Xtr, Utr, Ytr = jnp.asarray(X[tr]), jnp.asarray(U[tr]), jnp.asarray(Y[tr])
Xva, Uva, Yva = jnp.asarray(X[va]), jnp.asarray(U[va]), jnp.asarray(Y[va])
Xte, Ute, Yte = jnp.asarray(X[te]), jnp.asarray(U[te]), jnp.asarray(Y[te])

print("train:", Xtr.shape, Utr.shape, Ytr.shape)
print("val  :", Xva.shape, Uva.shape, Yva.shape)
print("test :", Xte.shape, Ute.shape, Yte.shape)

In [ ]:
def sample_batch(x, u, y, key):
    idx = jax.random.choice(key, x.shape[0], shape=(BATCH_SIZE,), replace=False)
    return x[idx], u[idx], y[idx]


def build_model(name):
    def forward(x, imu):
        if name == "imu_conditioned":
            cfg = fm.IMUConditionedConfig(
                vision_cfg=fm.ConvConfig(T=int(x.shape[1]), in_channels=int(x.shape[2]), hidden_channels=(24, 48), output_dim=64, dt=1.0),
                imu_dim=int(imu.shape[-1]), fused_dim=96, output_dim=3,
            )
            return fm.IMUConditionedVisualSNN(cfg)(x, imu)

        if name == "recurrent_fusion":
            cfg = fm.VisualIMURecurrentConfig(
                vision_cfg=fm.ConvConfig(T=int(x.shape[1]), in_channels=int(x.shape[2]), hidden_channels=(24, 48), output_dim=64, dt=1.0),
                imu_dim=int(imu.shape[-1]), hidden_dim=96, output_dim=3,
            )
            return fm.VisualIMURecurrentFusionBlock(cfg)(x, imu)

        if name == "kalman_style":
            cfg = fm.KalmanFusionConfig(
                vision_cfg=fm.ConvConfig(T=int(x.shape[1]), in_channels=int(x.shape[2]), hidden_channels=(24, 48), output_dim=64, dt=1.0),
                imu_dim=int(imu.shape[-1]), state_dim=32, output_dim=3,
            )
            return fm.KalmanStyleSpikingFusionSurrogate(cfg)(x, imu)

        raise ValueError(name)

    return hk.without_apply_rng(hk.transform(lambda x, imu: forward(x, imu)))


def train_model(name):
    transformed = build_model(name)
    params = transformed.init(jax.random.PRNGKey(SEED), Xtr[:BATCH_SIZE], Utr[:BATCH_SIZE])
    opt = optax.adam(1e-3)
    opt_state = opt.init(params)

    @jax.jit
    def step(params, opt_state, xb, ub, yb):
        def loss_fn(p):
            pred = transformed.apply(p, xb, ub)
            mse = jnp.mean((pred - yb) ** 2)
            return mse, pred

        (loss, pred), grads = jax.value_and_grad(loss_fn, has_aux=True)(params)
        updates, opt_state = opt.update(grads, opt_state, params)
        params = optax.apply_updates(params, updates)
        mae = jnp.mean(jnp.abs(pred - yb))
        return params, opt_state, loss, mae

    @jax.jit
    def eval_step(params, x, u, y):
        pred = transformed.apply(params, x, u)
        mse = jnp.mean((pred - y) ** 2)
        mae = jnp.mean(jnp.abs(pred - y))
        return mse, mae

    key = jax.random.PRNGKey(SEED + 99)
    for epoch in range(EPOCHS):
        key, sk = jax.random.split(key)
        xb, ub, yb = sample_batch(Xtr, Utr, Ytr, sk)
        params, opt_state, tr_mse, tr_mae = step(params, opt_state, xb, ub, yb)
        va_mse, va_mae = eval_step(params, Xva, Uva, Yva)
        print(f"{name} | epoch {epoch+1:02d} | train mse={tr_mse:.4f} mae={tr_mae:.4f} | val mse={va_mse:.4f} mae={va_mae:.4f}")

    te_mse, te_mae = eval_step(params, Xte, Ute, Yte)
    return float(te_mse), float(te_mae)


results = {}
for model_name in ["imu_conditioned", "recurrent_fusion", "kalman_style"]:
    mse, mae = train_model(model_name)
    results[model_name] = {"test_mse": mse, "test_mae": mae}

results